**Author**: Felipe Matheus
**Purpose**: Experiment launcher for the annealing **tensile strength (UTS)** surrogate.

Same architecture as `run_experiments.ipynb` (IACS): all pipeline logic lives
in `src/modeling/Experiments.py`, which is process-agnostic — the SAME
`ExperimentRunner` is reused; only the `ExperimentConfig` changes (target,
features, physical bounds). No new class needed.

Results layout: `models/annealing_tensile_strength/experiments/`
(`experiments_log.csv` + one folder per run).

# 1. Setup

In [1]:
import logging
import os
import sys

import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from autogluon.tabular import TabularPredictor

module_path = os.path.abspath(os.path.join('../..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from src.processing.Processing import Processing
from src.feature_engineering.FeatureEngineering import FeatureEngineering
from src.modeling.Modeling import Modeling
from src.metrics.Evaluation import Evaluation
from src.modeling.Experiments import ExperimentConfig, ExperimentRunner

from config.Variables import Variables

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)

%load_ext autoreload
%autoreload 2

varv = Variables()
proc = Processing()
feng = FeatureEngineering()
modl = Modeling()
evla = Evaluation()
runr = ExperimentRunner(modl, evla, models_root=varv.PATHS.models)

c:\Users\fmfoa\Projects\uncertainty-aware-predictors\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
TARGET = "tensile_strength_final"
ALL_FEATURES = ["purity", "initial_diameter", "tensile_strength", "temperature", "time"]
PROCESS = "annealing_uts"

SCHEMA_DATE = "110826"
FILE_NAME_SCHEMA_DATA = "schema_annealing_essays_{}.csv".format(SCHEMA_DATE)

TAG = f"annealing-schema{SCHEMA_DATE}"
GRID = {
    "time_limit_a": [900, 600],
    # "num_bag_folds_a": [10, 15, 20],
    # "features": [
    #     ("purity", "initial_diameter", "tensile_strength", "temperature", "time"),
    #     # ("initial_diameter", "tensile_strength", "temperature", "time"),
    # ],
}

# 2. Data (same preparation as annealing_uts.ipynb, run once)

In [ ]:
df, df_val = proc.process_annealing_uts(
    features=ALL_FEATURES,
    target=TARGET,
    df_schema=pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME_SCHEMA_DATA)),
)

In [ ]:
# FILE_NAME = "dataset_annealing_tensile-strength.csv"

# df_raw = pd.read_csv(os.path.join(varv.PATHS.data_raw, FILE_NAME))
# df_float = proc.df_to_float(
#     df_raw, drop_cols=["DOI", "is_Cu"], ignore_columns=["material"]
# )
# df_labeled = feng.label_element(df_float).drop_duplicates()
# df_with_masks = feng.add_ratio_mask_column(
#     feng.add_ratio_mask_column(df_labeled, "grain_size"), "tensile_strength",
# )
# df = (
#     df_with_masks[ALL_FEATURES + [TARGET]]
#     .dropna(subset=[TARGET])
#     .reset_index(drop=True)
# )

# # No essay rows yet for UTS -> no is_essay column. The runr detects this
# # and applies uniform weights (weight_on_essay_rows must stay 1.0).
# print(f"Dataset: {df.shape}")

# # Validation set: none held-out yet. When UTS essays arrive, build df_val
# # from them (with the same columns) and pass df_val=df_val below.
# df_val = None
# df.head()

Dataset: (92, 6)


c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:32: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(
c:\Users\fmfoa\Projects\uncertainty-aware-predictors\src\processing\Processing.py:37: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df_float = df_float.applymap(


,purity,initial_diameter,tensile_strength,temperature,time,tensile_strength_final
0,99.99,23.0,1200.0,373.0,60.0,1180.0
1,99.99,23.0,1200.0,473.0,60.0,1150.0
2,99.99,23.0,1200.0,573.0,60.0,850.0
3,99.99,23.0,1200.0,673.0,60.0,600.0
4,99.99,23.0,1200.0,773.0,60.0,330.0


# 3. Base config

In [4]:
base = ExperimentConfig(
    process=PROCESS,
    tag=TAG,
    target=TARGET,
    features=tuple(ALL_FEATURES),
    y_max=None,
    y_min=0.0,
    presets_a = "best_quality",
)
print(base.run_id)

annealing-uts-v2-best_quality__efda9a89


# 4. Single run (sanity check before any grid)

Run the base config alone first; then repeat 2-3x with `tag="uts-v1-rep2"`
etc. to measure run-to-run noise (the floor below which grid differences
mean nothing).

In [ ]:
result = runr.run_experiment(df, cfg=base, df_val=df_val)
result["artifacts"]["metrics"]

# 5 Validation set

TBD

In [ ]:
df_val

,initial_diameter,tensile_strength,purity,elongation,temperature,time,elongation_final
9,1.20,431.58,99.9000,0.94,623.0,60.0,26.17
25,2.00,376.08,99.9558,5.33,573.0,30.0,8.55
8,1.20,431.58,99.9000,0.94,623.0,30.0,27.06
21,1.20,335.21,99.9800,1.61,523.0,30.0,51.01
0,2.16,391.69,99.9000,4.90,573.0,30.0,48.60
12,1.20,397.09,99.9000,2.36,573.0,30.0,44.18
17,1.20,397.09,99.9000,2.36,523.0,90.0,42.28
22,1.20,335.21,99.9800,1.61,523.0,60.0,49.42


In [ ]:
result["artifacts"]["validation_metrics"]

{'rmse': 10.597993658519528,
 'mae': 6.547539176124967,
 'mape': 48.52385742130462,
 'r2': 0.43481691755998053,
 'coverage': {0.5: 0.25, 0.8: 0.75, 0.9: 0.75, 0.95: 0.875}}

In [ ]:
result["artifacts"].keys()

dict_keys(['config', 'features', 'target', 'base_model_names', 'weights', 'nnls_recovery_ok', 'nnls_max_diff', 'variance_floor', 'recalibration_c', 'ood_ref', 'y_max', 'y_min', 'calibration_before', 'calibration_after', 'aleatoric_diagnostics', 'metrics', 'validation_metrics', 'dataset_hash'])

In [ ]:
result.keys()

dict_keys(['cfg', 'run_dir', 'artifacts', 'log_row', 'predictor_a', 'predictor_b'])

In [ ]:
result

{'cfg': ExperimentConfig(process='annealing_elongation', tag='annealing-schema230726', base_tag=None, target='elongation_final', features=('initial_diameter', 'tensile_strength', 'purity', 'elongation', 'temperature', 'time'), weight_on_essay_rows=1.0, presets_a='medium_quality', num_bag_folds_a=5, num_bag_sets_a=1, num_stack_levels_a=0, time_limit_a=120, presets_b='medium_quality', num_bag_folds_b=5, num_stack_levels_b=0, time_limit_b=60, use_weighted_variance=True, variance_floor_frac=0.01, recalibration_target_alpha=0.9, calibration_alphas=(0.5, 0.8, 0.9, 0.95), y_max=None, y_min=None, fold_seed=42, use_shared_folds=False, group_col=None),
 'run_dir': Path('../../models/annealing_elongation/experiments/annealing-schema230726/annealing-schema230726__78b06c52'),
 'artifacts': {'config': {'process': 'annealing_elongation',
   'tag': 'annealing-schema230726',
   'base_tag': None,
   'target': 'elongation_final',
   'features': ['initial_diameter',
    'tensile_strength',
    'purity',
 

# 6. Grid

In [6]:
log = runr.run_grid(df, base_cfg=base, grid=GRID, df_val=df_val)
log

2026-06-25 23:32:35,855 | INFO | src.modeling.Experiments | Grid: 9 runs over ['time_limit_a', 'num_bag_folds_a']
2026-06-25 23:32:35,855 | INFO | src.modeling.Experiments | === Running annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=10__cc94c485 ===
2026-06-25 23:32:35,855 | INFO | src.modeling.Experiments | No 'is_essay' column: dataset has no essay rows; uniform weights applied.
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.11.9
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          8
Pytorch Version:    2.9.1+cpu
CUDA Version:       CUDA is not available
Memory Avail:       15.42 GB / 31.57 GB (48.8%)
Disk Space Avail:   740.19 GB / 932.08 GB (79.4%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Stack configuration (auto_stack=True): num_stack_levels=0, num_bag_folds=10, num_bag_sets=1

,run_id,timestamp,elapsed_s,n_rows,dataset_hash,model_dir,cfg_process,cfg_tag,cfg_base_tag,cfg_target,...,r2,cov_0.5,cov_0.8,cov_0.9,cov_0.95,c_opt,pct_truncated_aleat,nnls_recovery_ok,mean_sigma_epist,mean_sigma_aleat
0,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=10__cc94c485,2026-06-25T23:35:56,200.8,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=10__cc94c485,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=10,annealing-uts-v2-best_quality,tensile_strength_final,...,0.96933,0.5326,0.7609,0.8913,0.9565,0.8929,48.91,True,21.86564,19.08715
1,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=15__a7172634,2026-06-25T23:39:28,211.4,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=15__a7172634,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=15,annealing-uts-v2-best_quality,tensile_strength_final,...,0.97549,0.8043,0.9891,0.9891,1.0000,2.0451,41.30,True,17.04760,19.08715
2,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=20__14ac7284,2026-06-25T23:42:59,211.0,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=20__14ac7284,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=20,annealing-uts-v2-best_quality,tensile_strength_final,...,0.97185,0.4348,0.7826,0.9022,0.9783,1.1609,16.30,True,10.14668,19.08715
3,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=10__df848c54,2026-06-25T23:58:49,950.0,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=10__df848c54,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=10,annealing-uts-v2-best_quality,tensile_strength_final,...,0.97502,0.5109,0.8370,0.9239,0.9348,0.8647,52.17,True,18.67531,19.08715
4,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=15__5ac7f3e6,2026-06-26T00:14:42,953.4,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=15__5ac7f3e6,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=15,annealing-uts-v2-best_quality,tensile_strength_final,...,0.98018,0.4783,0.7609,0.9022,0.9239,0.8940,38.04,True,15.77075,19.08715
5,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=20__b6785ca4,2026-06-26T00:31:11,989.4,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=20__b6785ca4,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=20,annealing-uts-v2-best_quality,tensile_strength_final,...,0.98214,0.7283,0.9891,0.9891,0.9891,1.4549,43.48,True,16.20085,19.08715
6,annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=10__9f45055c,2026-06-26T08:20:48,28176.5,92,d26194f9,C:\Users\fmfoa\Projects\uncertainty-aware-predictors\models\annealing_uts\experiments\annealing-uts-v2-best_quality\annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=10__9f45055c,annealing_uts,annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=10,annealing-uts-v2-best_quality,tensile_strength_final,...,0.97641,0.5543,0.8152,0.9022,0.9565,0.9327,52.17,True,21.78419,19.08715
7,annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=15__a497979d,2026-06-26T09:54:53,5645.1,92,d261

# 7. Inspect results

In [7]:
log = runr.load_log(base)

view_cols = [
    "run_id", "cfg_time_limit_a", "cfg_num_bag_folds_a", "cfg_features",
    "rmse", "mae", "cov_0.9", "val_rmse", "val_mae", "val_cov_0.9",
    "c_opt", "pct_truncated_aleat", "mean_sigma_epist", "mean_sigma_aleat",
    "elapsed_s",
]
log[[c for c in view_cols if c in log.columns]].sort_values("rmse")

,run_id,cfg_time_limit_a,cfg_num_bag_folds_a,cfg_features,rmse,mae,cov_0.9,c_opt,pct_truncated_aleat,mean_sigma_epist,mean_sigma_aleat,elapsed_s
8,annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=20__394711fd,1800,20,purity|initial_diameter|tensile_strength|temperature|time,24.22465,17.39393,1.0000,3.0000,53.26,17.03703,19.08715,1851.3
5,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=20__b6785ca4,900,20,purity|initial_diameter|tensile_strength|temperature|time,25.37091,19.11637,0.9891,1.4549,43.48,16.20085,19.08715,989.4
4,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=15__5ac7f3e6,900,15,purity|initial_diameter|tensile_strength|temperature|time,26.72572,20.39585,0.9022,0.8940,38.04,15.77075,19.08715,953.4
6,annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=10__9f45055c,1800,10,purity|initial_diameter|tensile_strength|temperature|time,29.15651,21.87396,0.9022,0.9327,52.17,21.78419,19.08715,28176.5
1,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=15__a7172634,150,15,purity|initial_diameter|tensile_strength|temperature|time,29.72090,21.62264,0.9891,2.0451,41.30,17.04760,19.08715,211.4
3,annealing-uts-v2-best_quality__time_limit_a=900__num_bag_folds_a=10__df848c54,900,10,purity|initial_diameter|tensile_strength|temperature|time,30.00073,20.08160,0.9239,0.8647,52.17,18.67531,19.08715,950.0
2,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=20__14ac7284,150,20,purity|initial_diameter|tensile_strength|temperature|time,31.84747,23.87734,0.9022,1.1609,16.30,10.14668,19.08715,211.0
0,annealing-uts-v2-best_quality__time_limit_a=150__num_bag_folds_a=10__cc94c485,150,10,purity|initial_diameter|tensile_strength|temperature|time,33.24733,23.09902,0.8913,0.8929,48.91,21.86564,19.08715,200.8
7,annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=15__a497979d,1800,15,purity|initial_diameter|tensile_strength|temperature|time,70.34474,44.90797,0.9022,1.7072,0.00,0.00000,30.14455,5645.1


In [8]:
log.groupby("cfg_time_limit_a")[["rmse", "mae", "cov_0.9"]].agg(["mean", "std"])

rmse                   mae              cov_0.9  \
                       mean        std       mean        std      mean   
cfg_time_limit_a                                                         
150               31.605233   1.775651  22.866333   1.145218  0.927533   
900               27.365787   2.380351  19.864607   0.666770  0.938400   
1800              41.241967  25.324087  28.058620  14.762897  0.934800   

                            
                       std  
cfg_time_limit_a            
150               0.053596  
900               0.045228  
1800              0.056465

# 8. Load a winner

In [9]:
best_row = log.sort_values("rmse").iloc[0]
RUN_ID = best_row["run_id"]
run_dir = Path(best_row["model_dir"])   # <-- absolute path, base_tag included

with open(run_dir / "artifacts.pkl", "rb") as f:
    art = pickle.load(f)
predictor_a = TabularPredictor.load(str(run_dir / "model_a"))
predictor_b = TabularPredictor.load(str(run_dir / "model_b"))

print(RUN_ID)
art["calibration_after"]

annealing-uts-v2-best_quality__time_limit_a=1800__num_bag_folds_a=20__394711fd


,alpha,empirical_coverage,gap
0,0.50,1.0,0.50
1,0.80,1.0,0.20
2,0.90,1.0,0.10
3,0.95,1.0,0.05
